# Random Forests: Why Ask One Tree When You Can Ask Many?

**Session 3:** logistic regression draws one straight boundary.
**Session 4:** a single decision tree can carve out any-shaped region -- but an unconstrained tree just
memorizes the training data.
**This session:** what if we stopped trusting any one tree's judgment call?

In [1]:
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split

pd.set_option("display.precision", 3)

Last session ended with an unconstrained decision tree that scored a perfect 1.000 on training data -- and clearly worse on data it hadn't seen. Let's rebuild that exact tree and look at its region again.

<details>
<summary>Show code</summary>

```python
pass_df = pd.read_csv("../data/exam_pass_3d.csv")
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

deep_tree_model = DecisionTreeClassifier(max_depth=None, random_state=0)
deep_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {deep_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {deep_tree_model.score(X_test, y_test):.3f}")

grid_hours = np.linspace(X["hours_studied"].min(), X["hours_studied"].max(), 200)
grid_practice = np.linspace(X["practice_problems"].min(), X["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
deep_tree_grid = deep_tree_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=deep_tree_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Last Session's Unconstrained Tree (jagged, overfit)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()
```

</details>

In [2]:
pass_df = pd.read_csv("../data/exam_pass_3d.csv")
X = pass_df[["hours_studied", "practice_problems"]]
y = pass_df["passed"]
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)

deep_tree_model = DecisionTreeClassifier(max_depth=None, random_state=0)
deep_tree_model.fit(X_train, y_train)
print(f"Train accuracy: {deep_tree_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {deep_tree_model.score(X_test, y_test):.3f}")

grid_hours = np.linspace(X["hours_studied"].min(), X["hours_studied"].max(), 200)
grid_practice = np.linspace(X["practice_problems"].min(), X["practice_problems"].max(), 200)
grid_x, grid_y = np.meshgrid(grid_hours, grid_practice)
grid_points = pd.DataFrame({"hours_studied": grid_x.ravel(), "practice_problems": grid_y.ravel()})
deep_tree_grid = deep_tree_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=deep_tree_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Last Session's Unconstrained Tree (jagged, overfit)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()

Train accuracy: 1.000
Test accuracy:  0.879


## The Core Idea: Many Slightly-Different Trees

Imagine instead of asking one tree, we trained 100 trees, each on its own randomly-drawn version of the same
class roster -- some students appear twice, some don't appear at all -- and then let them vote on every
prediction. Any one tree's blind spot is unlikely to be shared by all 100 at once. This "randomly-drawn version"
is called a **bootstrap sample**: drawing rows with replacement from the original data, up to the original size.

> ### 🙋 Ask the class
>
> - If 100 people each guess a number independently and you average their guesses, why does the average often land closer to the truth than any individual guess?

## Bootstrap Sampling, Visualized

<details>
<summary>Show code</summary>

```python
rng = np.random.default_rng(seed=0)
sample_slice = pass_df.iloc[:8].reset_index(drop=True)
print("Original roster indices:", list(sample_slice.index))

for resample_number in range(1, 4):
    resample_indices = rng.choice(sample_slice.index, size=len(sample_slice), replace=True)
    print(f"Bootstrap resample {resample_number}:", sorted(resample_indices))
```

</details>

In [3]:
rng = np.random.default_rng(seed=0)
sample_slice = pass_df.iloc[:8].reset_index(drop=True)
print("Original roster indices:", list(sample_slice.index))

for resample_number in range(1, 4):
    resample_indices = rng.choice(sample_slice.index, size=len(sample_slice), replace=True)
    print(f"Bootstrap resample {resample_number}:", sorted(resample_indices))

Original roster indices: [0, 1, 2, 3, 4, 5, 6, 7]
Bootstrap resample 1: [np.int64(0), np.int64(0), np.int64(0), np.int64(2), np.int64(2), np.int64(4), np.int64(5), np.int64(6)]
Bootstrap resample 2: [np.int64(1), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(6), np.int64(7), np.int64(7)]
Bootstrap resample 3: [np.int64(0), np.int64(2), np.int64(4), np.int64(4), np.int64(5), np.int64(5), np.int64(6), np.int64(7)]


Notice some indices repeat within a resample, and some are missing entirely -- each tree in the forest will be trained on a roster that looks a little different from the original, and different from every other tree's roster.

## Fitting a Random Forest

<details>
<summary>Show code</summary>

```python
forest_model = RandomForestClassifier(n_estimators=100, random_state=0)
forest_model.fit(X_train, y_train)
print(f"Train accuracy: {forest_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {forest_model.score(X_test, y_test):.3f}")

forest_grid = forest_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=forest_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Random Forest Region (100 trees, averaged)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()
```

</details>

In [4]:
forest_model = RandomForestClassifier(n_estimators=100, random_state=0)
forest_model.fit(X_train, y_train)
print(f"Train accuracy: {forest_model.score(X_train, y_train):.3f}")
print(f"Test accuracy:  {forest_model.score(X_test, y_test):.3f}")

forest_grid = forest_model.predict(grid_points).reshape(grid_x.shape)

region_figure = go.Figure()
region_figure.add_trace(
    go.Contour(
        x=grid_hours, y=grid_practice, z=forest_grid,
        showscale=False, colorscale=[[0, "rgba(220,38,38,0.15)"], [1, "rgba(22,163,74,0.15)"]],
        contours=dict(start=0, end=1, size=1, coloring="fill"),
        line=dict(width=0), hoverinfo="skip",
    )
)
for passed_value, color, label in [(0, "#dc2626", "actual: failed"), (1, "#16a34a", "actual: passed")]:
    subset = pass_df[pass_df["passed"] == passed_value]
    region_figure.add_trace(
        go.Scatter(
            x=subset["hours_studied"], y=subset["practice_problems"], mode="markers",
            marker=dict(size=9, color=color, line=dict(width=1, color="white")), name=label,
        )
    )
region_figure.update_layout(
    title="Random Forest Region (100 trees, averaged)",
    xaxis_title="Hours studied", yaxis_title="Practice problems",
    template="plotly_white", width=750, height=550,
)
region_figure.show()

Train accuracy: 1.000
Test accuracy:  0.939


Same training accuracy as the single deep tree -- but noticeably better on data it never saw, and the region itself looks smoother, less jagged.

## Does the Forest Really Overfit Less?

<details>
<summary>Show code</summary>

```python
comparison_table = pd.DataFrame([
    {
        "model": "Single deep tree (max_depth=None)",
        "train_accuracy": deep_tree_model.score(X_train, y_train),
        "test_accuracy": deep_tree_model.score(X_test, y_test),
    },
    {
        "model": "Random forest (100 trees)",
        "train_accuracy": forest_model.score(X_train, y_train),
        "test_accuracy": forest_model.score(X_test, y_test),
    },
])
comparison_table
```

</details>

In [5]:
comparison_table = pd.DataFrame([
    {
        "model": "Single deep tree (max_depth=None)",
        "train_accuracy": deep_tree_model.score(X_train, y_train),
        "test_accuracy": deep_tree_model.score(X_test, y_test),
    },
    {
        "model": "Random forest (100 trees)",
        "train_accuracy": forest_model.score(X_train, y_train),
        "test_accuracy": forest_model.score(X_test, y_test),
    },
])
comparison_table

,model,train_accuracy,test_accuracy
0,Single deep tree (max_depth=None),1.0,0.879
1,Random forest (100 trees),1.0,0.939


> ### 🙋 Ask the class
>
> - The forest's training accuracy isn't lower than the single tree's -- so why is its test accuracy higher?

## How Many Trees Do We Need?

<details>
<summary>Show code</summary>

```python
tree_counts = [1, 2, 5, 10, 25, 50, 100, 200]
sweep_accuracies = []
for n in tree_counts:
    sweep_model = RandomForestClassifier(n_estimators=n, random_state=0)
    sweep_model.fit(X_train, y_train)
    sweep_accuracies.append(sweep_model.score(X_test, y_test))

sweep_figure = go.Figure()
sweep_figure.add_trace(
    go.Scatter(x=tree_counts, y=sweep_accuracies, mode="lines+markers",
               line=dict(color="#16a34a", width=3), marker=dict(size=9))
)
sweep_figure.update_layout(
    title="Test Accuracy vs. Number of Trees",
    xaxis_title="n_estimators", yaxis_title="test accuracy",
    template="plotly_white", width=700, height=450,
)
sweep_figure.show()
```

</details>

In [6]:
tree_counts = [1, 2, 5, 10, 25, 50, 100, 200]
sweep_accuracies = []
for n in tree_counts:
    sweep_model = RandomForestClassifier(n_estimators=n, random_state=0)
    sweep_model.fit(X_train, y_train)
    sweep_accuracies.append(sweep_model.score(X_test, y_test))

sweep_figure = go.Figure()
sweep_figure.add_trace(
    go.Scatter(x=tree_counts, y=sweep_accuracies, mode="lines+markers",
               line=dict(color="#16a34a", width=3), marker=dict(size=9))
)
sweep_figure.update_layout(
    title="Test Accuracy vs. Number of Trees",
    xaxis_title="n_estimators", yaxis_title="test accuracy",
    template="plotly_white", width=700, height=450,
)
sweep_figure.show()

> ### 🧑‍🏫 Instructor note
>
> Contrast with last session: more DEPTH could actively hurt test accuracy (the unlimited-depth tree). More
> TREES essentially never hurts accuracy here -- it just costs more compute past a certain point. That's a
> genuinely different kind of knob.

## Feature Importance, Averaged Across the Forest

Back to all three features, including the noise column `sleep_hours`.

<details>
<summary>Show code</summary>

```python
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

single_tree_importance = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X3, y3).feature_importances_
forest_importance = RandomForestClassifier(n_estimators=100, random_state=0).fit(X3, y3).feature_importances_

importance_figure = go.Figure()
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=single_tree_importance, name="single tree", marker=dict(color="#dc2626")))
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=forest_importance, name="random forest", marker=dict(color="#16a34a")))
importance_figure.update_layout(
    barmode="group", title="Feature Importance: Single Tree vs. Random Forest",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=700, height=450,
)
importance_figure.show()
```

</details>

In [7]:
X3 = pass_df[["hours_studied", "practice_problems", "sleep_hours"]]
y3 = pass_df["passed"]

single_tree_importance = DecisionTreeClassifier(max_depth=3, random_state=0).fit(X3, y3).feature_importances_
forest_importance = RandomForestClassifier(n_estimators=100, random_state=0).fit(X3, y3).feature_importances_

importance_figure = go.Figure()
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=single_tree_importance, name="single tree", marker=dict(color="#dc2626")))
importance_figure.add_trace(go.Bar(x=list(X3.columns), y=forest_importance, name="random forest", marker=dict(color="#16a34a")))
importance_figure.update_layout(
    barmode="group", title="Feature Importance: Single Tree vs. Random Forest",
    xaxis_title="feature", yaxis_title="importance",
    template="plotly_white", width=700, height=450,
)
importance_figure.show()

The single tree gave `sleep_hours` essentially zero importance. The forest gives it a small but nonzero score. That's not a bug -- each split inside each tree only considers a random subset of features, so every so often a tree is forced to split on `sleep_hours` simply because the useful features weren't in its sampled subset for that split. It's the forest's version of Session 2's lesson: even a genuinely noisy feature can look slightly meaningful by chance.

## Recap

```text
one tree
  ↓  (flexible, but memorizes noise)
many trees, each on a bootstrap resample + a random feature subset per split
  ↓
average / vote across all of them
  ↓
individual mistakes cancel out -> smoother, more reliable predictions
```

**The whole course arc:** line → plane → many features → R² & p-values → logistic regression → decision
boundary → decision trees → random forests.

> One more idea worth knowing about: instead of building every tree independently and averaging, **boosting**
> builds trees one at a time, each one specifically correcting the previous trees' mistakes. That's genuinely
> more material -- a natural next session, not today's.